In [1]:
import os
import cv2
import glob
import json
import pickle
import random
import numpy as np
from tqdm import tqdm
import random
random.seed(20)

In [2]:
LABELS_M2 = [
    "R1", "R2", "R3", "R4", # crops
    "RM1", "RM2",
    "RSL1", "RSM1",
    "LTE", "MTE", # edges
    "LTL", "MTM",
    "LTC", "MTC", # center
    "LTM", "MTL"
]

LABELS_M = [
    "RSLT1", "RSMT1" # low
]

def get_keypoints(filepath, LABELS):
    keypoints = {}

    with open(filepath, "r") as f:
        text = f.read()
    f.close()

    for line in text.split("\n"):
        if any(line.startswith(l) for l in LABELS):
            items = line.split()
            keypoints[items[0]] = (float(items[1]), float(items[3]))
    
    return keypoints

In [3]:
folderpaths = glob.glob("/dataset/*/*")

count = 0

data = []

for path in folderpaths:

    jpg_files_m = glob.glob(os.path.join(path, "M" , "*.jpg"))
    txt_files_m = glob.glob(os.path.join(path, "M" , "*.txt"))

    jpg_files_m2 = glob.glob(os.path.join(path, "M2" , "*.jpg"))
    txt_files_m2 = glob.glob(os.path.join(path, "M2" , "*.txt"))

    if len(jpg_files_m) > 1 or len(jpg_files_m2) > 1:
        print(f"Found multiple jpg files -> {path}")
    if len(txt_files_m) > 1 or len(txt_files_m2) > 1:
        print(f"Found multiple txt files: {path}")
    
    try:
        img_path = jpg_files_m2[0]
        keypoints_m = get_keypoints(txt_files_m[0], LABELS_M)
        keypoints_m2 = get_keypoints(txt_files_m2[0], LABELS_M2)
        
        for l in LABELS_M:
            if l not in keypoints_m:
                print(f"{l} is missing: {path}")
        for l in LABELS_M2:
            if l not in keypoints_m2:
                print(f"{l} is missing: {path}")
        
        keypoints_m2.update(keypoints_m)
        data.append((img_path, keypoints_m2))
    except:
        print(f"Something wrong: {path}")

print(f"data size: {len(data)}")

data size: 3168


In [4]:
random.shuffle(data)

In [5]:
total = len(data)
train_end = int(total * 0.8)
val_end = int(total * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

In [6]:
print(f"{len(train_data)}")
print(f"{len(val_data)}")
print(f"{len(test_data)}")

2534
317
317


In [8]:
# with open("pickle/train.pkl", "wb") as f:
#     pickle.dump(train_data, f)

# with open("pickle/val.pkl", "wb") as f:
#     pickle.dump(val_data, f)

# with open("pickle/test.pkl", "wb") as f:
#     pickle.dump(test_data, f)